In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import RobustScaler

from sklearn.model_selection import train_test_split,StratifiedKFold, GridSearchCV
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import randint

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV


import warnings

# Suppress FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
file_path = r'/Users/joelsabiti/Desktop/practicals/cereals/data.csv'

df = pd.read_csv(file_path)

In [ ]:
print(df.dtypes)

In [ ]:
duplicates = df[df.duplicated()]
print("Duplicated Rows:")
print(duplicates)

In [ ]:
df['type'].unique()

In [ ]:
type_counts = df['type'].value_counts()
plt.figure(figsize=(8, 6))
sns.barplot(x=type_counts.index, y=type_counts.values, palette='viridis')
plt.title('Distribution of Transaction Types')
plt.xlabel('Transaction Type')
plt.ylabel('Count')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
fraud_counts = df['isFraud'].value_counts()

plt.figure(figsize=(8, 6))
ax = sns.barplot(x=fraud_counts.index, y=fraud_counts.values, palette='viridis')
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=12, color='black')
ax.set_yscale('log')
plt.title('Distribution of Fraudulent Transactions')
plt.xlabel('Fraud Status')
plt.ylabel('Count (log scale)')
plt.show()

In [ ]:
result = df.groupby('isFraud')['type'].value_counts()
print(result)

In [ ]:

len(df['nameOrig'].unique())

In [ ]:
len(df['nameDest'].unique())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df['isFraud'] = df['isFraud'].astype(str)

plt.figure(figsize=(12, 8))

# Countplot for 'type' column with hue='isFraud' and logarithmic scale
ax = sns.countplot(x='type', hue='isFraud', data=df, palette='Set2')

# Adding value annotations on top of the bars
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=10, color='black')

# Set the y-axis to logarithmic scale
plt.yscale('log')

plt.title('Distribution of Transaction Types by Fraud Status')
plt.xlabel('Transaction Type')
plt.ylabel('Count (log scale)')

plt.show()
 

In [ ]:
df['type'] = df['type'].replace(['CASH_IN','PAYMENT','DEBIT'], 'OTHER')

In [ ]:
df.replace(to_replace = ['TRANSFER', 'CASH_OUT', 'OTHER'], value = [1,2,3],inplace = True)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
cols_to_drop=['nameDest','nameOrig','type']
df=df.drop(cols_to_drop, axis = 1)
spearman_corr_matrix = df.corr(method='spearman')
plt.figure(figsize=(10, 8))
sns.heatmap(spearman_corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, mask=spearman_corr_matrix.isna())
plt.title("Spearman's Rank Correlation Heatmap")
plt.show()


In [ ]:
df['step'] = df['step'] / 24
df['step'] = df['step'].round().astype(int)
df['day'] = df['step'] % 7
day_mapping = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
df['day'] = df['day'].map(day_mapping)

In [ ]:
custom_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ordinal_encoder = OrdinalEncoder(categories=[custom_order])
df['day'] = ordinal_encoder.fit_transform(df[['day']])

In [ ]:
df

In [ ]:
cols_to_drop=['oldbalanceDest','newbalanceDest','newbalanceOrig','isFlaggedFraud','step']
df.drop(cols_to_drop, axis=1, inplace=True)

In [ ]:
df['isFraud'] = df['isFraud'].astype(int)

# Models Without resampling

## XG BOOST Classifier

In [ ]:
from sklearn.metrics import f1_score
X = df.drop('isFraud', axis=1) 
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])

X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model = xgb.XGBClassifier(objective="binary:logistic", random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

report = classification_report(y_test, y_pred,digits=6)
print("Classification Report:\n", report)

f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")


## Random Forest 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
X = df.drop('isFraud', axis=1) 
y = df['isFraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model =RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nRandom Forest Results using SMOTE:")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

report = classification_report(y_test, y_pred,digits=6)
print("Classification Report:\n", report)

f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")

## Decision Tree Classifier 

In [ ]:
X = df.drop('isFraud', axis=1) 
y = df['isFraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nDecision Trees using SMOTE TOMEK:")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred,digits=4))
f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")

## KNN 

In [ ]:
X = df.drop('isFraud', axis=1) 
y = df['isFraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model = KNeighborsClassifier(n_neighbors=20)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nKNN Results using SMOTE TOMEK:")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred,digits=4))
f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")

# Models with SMOTE and SMOTE TOMEK Links

## XGB with smote

In [ ]:
from sklearn.metrics import f1_score
smt = SMOTE(random_state=42)
X = df.drop('isFraud', axis=1) 
y = df['isFraud']
X,y = smt.fit_resample(X,y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model = xgb.XGBClassifier(objective="binary:logistic", random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

report = classification_report(y_test, y_pred,digits=6)
print("Classification Report:\n", report)

f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")


## XGB with smote and TOMEK Links

In [ ]:
smt = SMOTETomek(random_state=2)
X = df.drop('isFraud', axis=1) 
y = df['isFraud']
X,y = smt.fit_resample(X,y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model = xgb.XGBClassifier(objective="binary:logistic", random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

report = classification_report(y_test, y_pred,digits=6)
print("Classification Report:\n", report)

f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")

## RANDOM FOREST

### Random Forest with SMOTE

In [ ]:
from sklearn.ensemble import RandomForestClassifier
smt = SMOTE(random_state=42)
X = df.drop('isFraud', axis=1) 
y = df['isFraud']

X, y = smt.fit_resample(X,y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model =RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nRandom Forest Results using SMOTE:")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

report = classification_report(y_test, y_pred,digits=6)
print("Classification Report:\n", report)

f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")

### Random Forest with SMOTE and TOMEK Links

In [ ]:
from sklearn.ensemble import RandomForestClassifier
smt = SMOTETomek(random_state=2)
X = df.drop('isFraud', axis=1) 
y = df['isFraud']
X, y = smt.fit_resample(X,y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model =RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred =model.predict(X_test)
print("\nRandom Forest Results using SMOTE:")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred,digits=4))
f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")

## Decision Trees

### Decision Trees with SMOTE and TOMEK Links

In [ ]:
smt = SMOTETomek(random_state=2)
X = df.drop('isFraud', axis=1) 
y = df['isFraud']
X, y = smt.fit_resample(X,y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nDecision Trees using SMOTE TOMEK:")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred,digits=4))


f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")

### Decision Trees with SMOTE

In [ ]:
smt = SMOTE(random_state=42)
X = df.drop('isFraud', axis=1) 
y = df['isFraud']
X, y = smt.fit_resample(X,y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nDecision Trees Results using SMOTE:")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred,digits=4))



f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")

## KNN

### KNN with SMOTE and TOMEK Links

In [ ]:
smt = SMOTETomek(random_state=2)
X = df.drop('isFraud', axis=1) 
y = df['isFraud']

X, y = smt.fit_resample(X,y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg']] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model = KNeighborsClassifier(n_neighbors=20)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nKNN Results using SMOTE TOMEK:")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred,digits=4))



f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")

### KNN with SMOTE

In [ ]:
smt = SMOTE(random_state=2)
X = df.drop('isFraud', axis=1) 
y = df['isFraud']

X, y = smt.fit_resample(X,y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
scaled_columns = scaler.fit_transform(X_train[['amount','oldbalanceOrg']])
X_train_scaled = X_train.copy()
X_train_scaled[['amount','oldbalanceOrg'] = scaled_columns
scaled_columns = scaler.fit_transform(X_test[['amount','oldbalanceOrg']])
X_test_scaled = X_test.copy()
X_test_scaled[['amount','oldbalanceOrg']] = scaled_columns
X_train = X_train_scaled
X_test = X_test_scaled
model = KNeighborsClassifier(n_neighbors=20)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nKNN Results using SMOTE:")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred,digits=4))



f1_minority = f1_score(y_test, y_pred, average='binary')
print(f"F1 Score for Minority Class: {f1_minority}")